In [1]:
import sympy as sp
import numpy as np
import scipy as sci

In [2]:
# define independent variables 
x,y = sp.symbols('x, y', real=True)
xv = sp.Matrix([x,y])

# define the rhs of the governing equation
w = sp.symbols("omega")
u1,u2 = sp.symbols("u_x, u_y", real=True)
u1 = sp.Function("u_x")(x, y)
u2 = sp.Function("u_y")(x, y)
u = sp.Matrix([u1,u2])
grad_w = sp.Matrix([sp.Derivative(w,x), sp.Derivative(w,y)])
FU = -u.dot(grad_w)
FU

-u_x(x, y)*Derivative(omega, x) - u_y(x, y)*Derivative(omega, y)

In [3]:
# define q(t)
N = 2 #number of vortexes

q = sp.Matrix()

A = sp.symbols("A", real=True)
L = sp.symbols("L", real=True, positive=True)

# xc= sp.symbols("x_c", real=True)
# yc= sp.symbols("y_c", real=True)

xc= sp.Matrix()
yc= sp.Matrix()
# r = sp.Matrix()

for i in range(N):
    xc = sp.Matrix([xc, sp.symbols("x_c_"+str(i+1), real=True)])
    yc = sp.Matrix([yc, sp.symbols("y_c_"+str(i+1), real=True)])
    # r = sp.Matrix([r, sp.symbols("r_"+str(i+1), real=True, positive=True)])
    # r = sp.Matrix([r, sp.Function("r_"+str(i+1))(x, y, xc[i], yc[i])])

q = sp.Matrix([A, L, xc, yc])
# qr = sp.Matrix([A, L, r])

q

Matrix([
[    A],
[    L],
[x_c_1],
[x_c_2],
[y_c_1],
[y_c_2]])

In [4]:
# define the ansatz_sym u_hat(x; q)
ansatz_gamma = 0
ansatz_gamma = A*sp.exp(-((x-xc[0])**2+(y-yc[0])**2)/L**2) + A*sp.exp(-((x-xc[1])**2+(y-yc[1])**2)/L**2)

ansatz_gamma

A*exp((-(x - x_c_1)**2 - (y - y_c_1)**2)/L**2) + A*exp((-(x - x_c_2)**2 - (y - y_c_2)**2)/L**2)

In [5]:
ansatz_u_sym = sp.Matrix([
    sp.Derivative(ansatz_gamma,y).doit().simplify(),
    -sp.Derivative(ansatz_gamma, x).doit().simplify()
])

ansatz_u_sym.simplify()
ansatz_u_sym

Matrix([
[-2*A*((y - y_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (y - y_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))/L**2],
[ 2*A*((x - x_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (x - x_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))/L**2]])

In [6]:
ansatz_sym = (- sp.Derivative(ansatz_gamma, x, 2) - sp.Derivative(ansatz_gamma, y, 2)).doit()
# ansatz_sym = (-sp.Derivative(ansatz_u_sym[0], y) ).doit() + sp.Derivative(ansatz_u_sym[1], x).doit()
ansatz_sym.simplify()

4*A*(L**2*(exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) + exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2)) - (x - x_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (x - x_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) - (y - y_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (y - y_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))/L**4

In [7]:
# compute partial derivatives du/dqi
dwdq_sym = ansatz_sym.diff(q)

dwdq_sym.simplify()

Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         4*(L**2*(exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) + exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2)) - (x - x_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (x - x_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) - (y - y_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (y - y_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))/L**4],
[-8*A*exp(-(x - x_c_1)**2/L**2 - (y - y_c_1)**2/L**2)/L**3 - 8*A*exp(-(x - x_c_2)**2/L**2 - (y 

In [8]:
dwdq_sym[0].simplify()

4*(L**2*(exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) + exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2)) - (x - x_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (x - x_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) - (y - y_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (y - y_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))/L**4

In [9]:
# data
a = 1
l = 1
xc1 = 1
yc1 = 1
xc2 = -1
yc2 = -1

def sub_data(f):
    return f.subs(A,a).subs(L,l).subs(xc[0],xc1).subs(yc[0],yc1).subs(xc[1],xc2).subs(yc[1],yc2)

In [10]:
dwdq = sub_data(dwdq_sym)
ansatz = sub_data(ansatz_sym)
ansatz_u = sub_data(ansatz_u_sym)

In [11]:
# Pre-allocate static integration grid ONCE
GRID_BOUND = 20.0
GRID_POINTS = 500  
grid_1d = np.linspace(-GRID_BOUND, GRID_BOUND, GRID_POINTS)
X_mesh, Y_mesh = np.meshgrid(grid_1d, grid_1d, indexing='ij')

In [12]:
def inner_prod_H(f, g, vars=(x, y)):
    """
    Computes numerical 2D integral of f * g over R^2 for SymPy expressions.
    """
    integrand_func = sp.lambdify(vars, (f*g), "numpy")

    Z = integrand_func(X_mesh, Y_mesh)

    # # To integrate x first, then y:
    int_x= sci.integrate.simpson(Z, x=grid_1d, axis=0)
    res =  sci.integrate.simpson(int_x, x=grid_1d, axis=0)

    # print(err1, err2)
    
    return res

In [13]:
# construct the matrix M_ij = <du/dqi, du/dqj>_H
n = len(q)
M = sp.zeros(n, n)

for i in range(n):
    M[i, i] = inner_prod_H(dwdq[i], dwdq[i])
    print(M[i, i])
    for j in range(i+1, n):
        M[i, j] = inner_prod_H(dwdq[i], dwdq[j])
        M[j,i] = M[i,j]
        print(M[i, j])


25.5930634413476
-27.4343522918644
0.460322212629207
-0.460322212629206
0.460322212629207
-0.460322212629206
95.0070983633229
2.76193327577524
-2.76193327577524
2.76193327577524
-2.76193327577524
37.6991118430775
1.61112774420222
-1.18650934754744e-16
1.84128885051683
37.6991118430775
1.84128885051683
2.46967083165317e-17
37.6991118430775
1.61112774420222
37.6991118430775


In [14]:
M

Matrix([
[  25.5930634413476, -27.4343522918644,     0.460322212629207,   -0.460322212629206,     0.460322212629207,   -0.460322212629206],
[ -27.4343522918644,  95.0070983633229,      2.76193327577524,    -2.76193327577524,      2.76193327577524,    -2.76193327577524],
[ 0.460322212629207,  2.76193327577524,      37.6991118430775,     1.61112774420222, -1.18650934754744e-16,     1.84128885051683],
[-0.460322212629206, -2.76193327577524,      1.61112774420222,     37.6991118430775,      1.84128885051683, 2.46967083165317e-17],
[ 0.460322212629207,  2.76193327577524, -1.18650934754744e-16,     1.84128885051683,      37.6991118430775,     1.61112774420222],
[-0.460322212629206, -2.76193327577524,      1.84128885051683, 2.46967083165317e-17,      1.61112774420222,     37.6991118430775]])

In [15]:
FU

-u_x(x, y)*Derivative(omega, x) - u_y(x, y)*Derivative(omega, y)

In [16]:
# compute rhs from the ansatz_sym
Fua = FU.subs(u1, ansatz_u[0]).subs(u2, ansatz_u[1]).subs(w, ansatz).doit()
Fua.simplify()

128*(-x**2 + y**2)*exp(-2*x**2 - 2*y**2 - 4)

In [17]:
# compute f
n = len(q)
f = sp.zeros(n, 1)

for i in range(n):
    f[i] = inner_prod_H(dwdq[i], Fua)
    print(f[i])

f

-6.93889390390723e-18
-5.55111512312578e-17
-0.639155188728404
0.639155188728404
0.639155188728404
-0.639155188728404


Matrix([
[-6.93889390390723e-18],
[-5.55111512312578e-17],
[   -0.639155188728404],
[    0.639155188728404],
[    0.639155188728404],
[   -0.639155188728404]])

In [18]:
q_dot = M.inv()*f

q_dot

Matrix([
[-2.38524477946811e-18],
[-1.19262238973405e-18],
[  -0.0168512375542028],
[   0.0168512375542028],
[   0.0168512375542028],
[  -0.0168512375542028]])